In [1]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

url = "https://okdiario.com/internacional/feed"
# url = "https://okdiario.com/espana/feed"
# url = "https://okdiario.com/cultura/feed"
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'xml')

items = soup.find_all('item')
print(f"Total items: {len(items)}")

Total items: 50


In [ ]:
datos = []

for item in items:
    # Extaer datos
    title = item.find("title").text if item.find("title") else None
    link = item.find("link").text if item.find("link") else None
    date = item.find("pubDate").text if item.find("pubDate") else None
    category = item.find("category").text if item.find("category") else None
    
    # Formatear datos
    formatted_date =datetime.strptime(date, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d") # Limpiar fecha

    # Elegir bien el contenido
    content_encoded_list = item.find_all("content:encoded")
    if content_encoded_list:
        html_content = content_encoded_list[-1].text
        soup_content = BeautifulSoup(html_content, 'html.parser')
        
        # Elimina los spans de comentarios
        for span in soup_content.find_all('span', class_='comment-text'):
            span.decompose()
        
        description = soup_content.get_text(strip=True)
    else:
        description = None



    # Guardar datos
    datos.append({
        "Link": link,
        "Periódico": "okdiario",
        "Fecha": formatted_date,
        "Título": title,
        "Subtítulo": None,
        "Categoría": category,
        "Contenido": description
    })

with open('data/okdiario.json', 'w', encoding='utf-8') as f:
    json.dump(datos, f, indent=2, ensure_ascii=False)